# Motor and VFD Analysis with NeqSim

Deep dive into the motor sizing, VFD topology selection, harmonic analysis,
and part-load performance modelling available in NeqSim's electrical design
framework.

**Topics covered:**
1. IEC motor sizing across power range — efficiency classes IE1–IE4
2. VFD topology selection (2-level, 3-level, multi-level, AFE)
3. Harmonic distortion (THD) analysis per IEEE 519
4. Part-load efficiency maps for motor + VFD
5. Cable sizing with derating factors
6. Hazardous area classification (IECEx/ATEX)
7. Combined motor + VFD efficiency curves at variable speed

In [ ]:
# Setup
import subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    if ns is not None:
        ns = neqsim_classes(ns)
        NEQSIM_MODE = "devtools"
        print("NeqSim loaded via devtools")
    else:
        raise RuntimeError("devtools returned None")
except Exception:
    try:
        import neqsim
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
    print("NeqSim loaded via pip")

import json
import jpype
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# Import electrical design components via jpype
ElectricalMotor = jpype.JClass("neqsim.process.electricaldesign.components.ElectricalMotor")
VFD = jpype.JClass("neqsim.process.electricaldesign.components.VariableFrequencyDrive")
Cable = jpype.JClass("neqsim.process.electricaldesign.components.ElectricalCable")
HazArea = jpype.JClass("neqsim.process.electricaldesign.components.HazardousAreaClassification")
Transformer = jpype.JClass("neqsim.process.electricaldesign.components.Transformer")
Switchgear = jpype.JClass("neqsim.process.electricaldesign.components.Switchgear")

print(f"Mode: {NEQSIM_MODE}")

## 1. Motor Sizing Across Power Range

Size motors from 10 kW to 5000 kW and compare the four IEC efficiency
classes (IE1 through IE4).

### IEC 60034-30-1 Motor Selection

The motor rated power is the IEC standard step ≥ required power × margin:

$$P_{\text{rated}} = \min\{P_{\text{std}} \ge P_{\text{shaft}} \times 1.10\}$$

Synchronous speed: $n_s = 120f/p$, rated speed: $n_r = n_s(1-s)$

In [ ]:
# Motor sizing across power range
shaft_powers = [10, 22, 45, 75, 110, 200, 315, 500, 800, 1250, 2000, 3500, 5000]
efficiency_classes = ["IE1", "IE2", "IE3", "IE4"]

results = {ec: {"rated": [], "eff": [], "pf": [], "current": [], "frame": [], "weight": []}
           for ec in efficiency_classes}

for ec in efficiency_classes:
    for sp in shaft_powers:
        motor = ElectricalMotor()
        motor.setEfficiencyClass(ec)
        motor.setPoles(4)
        motor.setRatedVoltageV(690.0 if sp > 75 else 400.0)
        motor.sizeMotor(float(sp), 1.10, "IEC")

        results[ec]["rated"].append(motor.getRatedPowerKW())
        results[ec]["eff"].append(motor.getEfficiencyPercent())
        results[ec]["pf"].append(motor.getPowerFactorFL())
        results[ec]["current"].append(motor.getRatedCurrentA())
        results[ec]["frame"].append(str(motor.getFrameSize()))
        results[ec]["weight"].append(motor.getWeightKg())

# Print table
print(f"{'Power':>8} {'Rated':>8} {'IE1 eff':>8} {'IE2 eff':>8} {'IE3 eff':>8} {'IE4 eff':>8} {'Frame':>8}")
print("-" * 60)
for i, sp in enumerate(shaft_powers):
    print(f"{sp:>8} {results['IE3']['rated'][i]:>8.0f}"
          f" {results['IE1']['eff'][i]:>7.1f}%"
          f" {results['IE2']['eff'][i]:>7.1f}%"
          f" {results['IE3']['eff'][i]:>7.1f}%"
          f" {results['IE4']['eff'][i]:>7.1f}%"
          f" {results['IE3']['frame'][i]:>8}")

In [ ]:
# Plot efficiency vs power for all efficiency classes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = {"IE1": "#e15759", "IE2": "#f28e2b", "IE3": "#59a14f", "IE4": "#4e79a7"}
for ec in efficiency_classes:
    ax1.plot(shaft_powers, results[ec]["eff"], "o-", color=colors[ec],
             label=f"{ec}", lw=2, markersize=5)

ax1.set_xlabel("Shaft Power (kW)")
ax1.set_ylabel("Efficiency (%)")
ax1.set_title("Motor Efficiency vs Power — IEC 60034-30-1")
ax1.set_xscale("log")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(78, 100)

# Power factor vs size
ax2.plot(shaft_powers, results["IE3"]["pf"], "s-", color="steelblue", lw=2, markersize=6)
ax2.axhline(0.85, color="gray", ls="--", alpha=0.5, label="cos\u03c6 = 0.85")
ax2.set_xlabel("Shaft Power (kW)")
ax2.set_ylabel("Power Factor")
ax2.set_title("Motor Power Factor vs Power (IE3, 4-pole)")
ax2.set_xscale("log")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. VFD Topology Selection

Size VFDs for motors at different power and voltage levels to see how
topology is automatically selected.

| Voltage | Power | Topology | THD |
|---------|-------|----------|-----|
| LV (≤690V) | ≤250 kW | 2-level, 6-pulse | 35% |
| LV (≤690V) | >250 kW | 2-level, AFE | 5% |
| MV (3.3kV) | ≤2 MW | 3-level, 12-pulse | 10% |
| MV (6.6kV+) | >2 MW | Multi-level, AFE | 3% |

In [ ]:
# VFD topology survey
test_cases = [
    ("Small LV",   45.0,   400.0),
    ("Medium LV",  200.0,  690.0),
    ("Large LV",   500.0,  690.0),
    ("Small MV",   800.0,  3300.0),
    ("Medium MV",  1600.0, 6600.0),
    ("Large MV",   3500.0, 6600.0),
    ("Very Large", 8000.0, 11000.0),
]

vfd_results = []
print(f"{'Case':<14} {'Power':>7} {'Voltage':>8} {'Topology':>12} {'Pulse':>8} "
      f"{'THD':>6} {'Eff':>6} {'PF':>6} {'Cool':>6}")
print("-" * 82)

for name, power, voltage in test_cases:
    motor = ElectricalMotor()
    motor.setRatedVoltageV(voltage)
    motor.sizeMotor(power, 1.10, "IEC")

    vfd = VFD()
    vfd.sizeVFD(motor)

    row = {
        "name": name,
        "power": motor.getRatedPowerKW(),
        "voltage": voltage,
        "topology": str(vfd.getTopologyType()),
        "pulse": str(vfd.getPulseConfiguration()),
        "thd": vfd.getThdCurrentPercent(),
        "eff": vfd.getEfficiencyPercent(),
        "pf": vfd.getInputPowerFactor(),
        "cool": str(vfd.getCoolingMethod()),
    }
    vfd_results.append(row)

    print(f"{name:<14} {row['power']:>7.0f} {voltage:>8.0f} {row['topology']:>12} "
          f"{row['pulse']:>8} {row['thd']:>5.1f}% {row['eff']:>5.1f}% "
          f"{row['pf']:>5.2f} {row['cool']:>6}")

In [ ]:
# THD and efficiency comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

names_vfd = [r["name"] for r in vfd_results]
thd_vals = [r["thd"] for r in vfd_results]
eff_vals = [r["eff"] for r in vfd_results]
x = np.arange(len(names_vfd))

# THD bar chart
bar_colors = ["red" if t > 10 else "orange" if t > 5 else "green" for t in thd_vals]
ax1.bar(x, thd_vals, color=bar_colors, edgecolor="white")
ax1.axhline(5, color="gray", ls="--", lw=1, label="IEEE 519 limit (5%)")
ax1.set_xticks(x)
ax1.set_xticklabels(names_vfd, rotation=30, ha="right", fontsize=9)
ax1.set_ylabel("THD Current (%)")
ax1.set_title("VFD Current THD by Configuration")
ax1.legend()
ax1.grid(True, alpha=0.3, axis="y")

for i, v in enumerate(thd_vals):
    ax1.text(i, v + 0.5, f"{v:.0f}%", ha="center", fontsize=9)

# Efficiency bar chart
ax2.bar(x, eff_vals, color="steelblue", edgecolor="white")
ax2.set_xticks(x)
ax2.set_xticklabels(names_vfd, rotation=30, ha="right", fontsize=9)
ax2.set_ylabel("Efficiency (%)")
ax2.set_title("VFD Efficiency by Configuration")
ax2.set_ylim(95, 99)
ax2.grid(True, alpha=0.3, axis="y")

for i, v in enumerate(eff_vals):
    ax2.text(i, v + 0.05, f"{v:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 3. Combined Motor + VFD Efficiency Map

At each speed and load, the combined efficiency is:

$$\eta_{\text{total}}(L, s) = \eta_{\text{motor}}(L) \times \eta_{\text{VFD}}(L, s)$$

This creates a 2D efficiency map showing where the drive is most/least
efficient.

In [ ]:
# Combined motor + VFD efficiency map
motor_500 = ElectricalMotor()
motor_500.setEfficiencyClass("IE3")
motor_500.setRatedVoltageV(690.0)
motor_500.sizeMotor(500.0, 1.10, "IEC")

vfd_500 = VFD()
vfd_500.sizeVFD(motor_500)

load_range = np.linspace(0.1, 1.2, 30)
speed_range = np.linspace(0.2, 1.0, 25)
L, S = np.meshgrid(load_range, speed_range)
eff_map = np.zeros_like(L)

for i in range(L.shape[0]):
    for j in range(L.shape[1]):
        motor_eff = motor_500.getEfficiencyAtLoad(float(L[i, j]))
        vfd_eff = vfd_500.getEfficiency(float(L[i, j]), float(S[i, j]))
        eff_map[i, j] = motor_eff * vfd_eff / 100.0  # combined %

fig, ax = plt.subplots(figsize=(10, 7))
cs = ax.contourf(L * 100, S * 100, eff_map, levels=20, cmap="RdYlGn")
plt.colorbar(cs, label="Combined Efficiency (%)")
ax.contour(L * 100, S * 100, eff_map, levels=[85, 88, 90, 92, 93],
           colors="black", linewidths=0.5)

ax.set_xlabel("Motor Load (%)")
ax.set_ylabel("Speed (%)")
ax.set_title(f"Combined Efficiency Map — {motor_500.getRatedPowerKW():.0f} kW "
             f"{motor_500.getEfficiencyClass()} Motor + "
             f"{vfd_500.getTopologyType()} VFD")
ax.grid(True, alpha=0.2)

# Mark design point
ax.plot(100, 100, "k*", markersize=15, label="Design point")
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

print(f"Motor: {motor_500.getRatedPowerKW():.0f} kW, {motor_500.getEfficiencyClass()}")
print(f"VFD: {vfd_500.getTopologyType()}, {vfd_500.getPulseConfiguration()}")
print(f"Design-point combined eff: {eff_map[-1, -1]:.1f}%")
print(f"Worst-case (20% speed, 10% load): {eff_map[0, 0]:.1f}%")

## 4. Cable Sizing with Derating

Cable cross-section is selected per IEC 60502 based on derated current:

$$I_{\text{required}} = \frac{I_{\text{load}}}{f_{\text{temp}} \times f_{\text{group}} \times f_{\text{depth}}}$$

Voltage drop:

$$\Delta V\% = \frac{\sqrt{3} \times I \times L \times (r\cos\varphi + x\sin\varphi)}{V} \times 100$$

In [ ]:
# Cable sizing across different conditions
cable_cases = [
    ("Case A: Short/Cool",   150.0, 400.0,  30.0, "Tray",     25.0),
    ("Case B: Short/Hot",    150.0, 400.0,  30.0, "Tray",     50.0),
    ("Case C: Long/Cool",    150.0, 400.0, 200.0, "Tray",     25.0),
    ("Case D: Long/Hot",     150.0, 400.0, 200.0, "Tray",     50.0),
    ("Case E: Conduit",      150.0, 400.0, 100.0, "Conduit",  40.0),
    ("Case F: Direct burial",150.0, 400.0, 100.0, "Direct burial", 40.0),
    ("Case G: Large/MV",     500.0, 6600.0, 150.0, "Tray",    40.0),
]

print(f"{'Case':<24} {'I(A)':>6} {'V':>6} {'L(m)':>6} {'Install':>14} "
      f"{'T(C)':>5} {'mm²':>5} {'Amp':>5} {'Vd%':>5} {'SC(kA)':>6}")
print("-" * 100)

cable_data = []
for name, current, voltage, length, method, temp in cable_cases:
    cable = Cable()
    cable.setLengthM(length)
    cable.sizeCable(current, voltage, length, method, temp)

    row = {
        "name": name,
        "mm2": cable.getCrossSectionMM2(),
        "amp": cable.getAmpacityA(),
        "vdrop": cable.getVoltageDropPercent(),
        "sc": cable.getShortCircuitWithstandKA(),
    }
    cable_data.append(row)

    print(f"{name:<24} {current:>6.0f} {voltage:>6.0f} {length:>6.0f} {method:>14} "
          f"{temp:>5.0f} {row['mm2']:>5.0f} {row['amp']:>5.0f} {row['vdrop']:>5.2f} "
          f"{row['sc']:>6.1f}")

In [ ]:
# Voltage drop vs cable length for a 150A, 400V load
lengths = np.arange(10, 501, 10)
vdrops = []
sizes = []

for length in lengths:
    cable = Cable()
    cable.setLengthM(float(length))
    cable.sizeCable(150.0, 400.0, float(length), "Tray", 40.0)
    vdrops.append(cable.getVoltageDropPercent())
    sizes.append(cable.getCrossSectionMM2())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(lengths, vdrops, "b-", lw=2)
ax1.axhline(5.0, color="red", ls="--", lw=1.5, label="Max 5% (IEC 60364)")
ax1.set_xlabel("Cable Length (m)")
ax1.set_ylabel("Voltage Drop (%)")
ax1.set_title("Voltage Drop vs Cable Length (150 A, 400 V, Tray)")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.step(lengths, sizes, "g-", lw=2, where="post")
ax2.set_xlabel("Cable Length (m)")
ax2.set_ylabel("Cable Size (mm\u00b2)")
ax2.set_title("Required Cable Size vs Length")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Hazardous Area Classification

Equipment in hydrocarbon service is classified per IEC 60079-10.
The classification determines required Ex protection level.

In [ ]:
# Classify different equipment types
equipment_types = [
    ("Compressor",     True,  120.0),
    ("Pump",           True,   80.0),
    ("Separator",      True,  100.0),
    ("HeatExchanger",  True,  150.0),
    ("Pipeline",       True,   60.0),
    ("Compressor",     False, 100.0),  # non-HC service
]

print(f"{'Equipment':<16} {'HC':>4} {'MaxT':>6} {'Zone':<12} {'TempClass':>10} {'EPL':>4} {'Ex Marking':<22}")
print("-" * 82)

for eq_type, has_hc, max_temp in equipment_types:
    haz = HazArea()
    haz.classify(eq_type, has_hc, max_temp)

    print(f"{eq_type:<16} {'Yes' if has_hc else 'No':>4} {max_temp:>6.0f} "
          f"{str(haz.getZone()):<12} {str(haz.getTemperatureClass()):>10} "
          f"{str(haz.getEquipmentProtectionLevel()):>4} {str(haz.getExMarking()):<22}")

## 6. Complete Drive System Analysis

Full analysis of a 500 kW compressor drive showing the entire
electrical chain — motor, VFD, cable, switchgear, transformer.

In [ ]:
# Full drive system for 500 kW compressor
shaft_power = 500.0  # kW
voltage = 690.0  # V
cable_length = 100.0  # m

# 1. Motor
motor = ElectricalMotor()
motor.setEfficiencyClass("IE3")
motor.setRatedVoltageV(voltage)
motor.sizeMotor(shaft_power, 1.10, "IEC")

# 2. VFD
vfd = VFD()
vfd.sizeVFD(motor)

# 3. Power calculations
motor_input = shaft_power / (motor.getEfficiencyPercent() / 100.0)
vfd_input = vfd.getElectricalInputKW(motor_input)
motor_loss = motor_input - shaft_power
vfd_loss = vfd_input - motor_input
total_loss = vfd_input - shaft_power

# 4. Current
pf = vfd.getInputPowerFactor()
fla = (vfd_input * 1000.0) / (3**0.5 * voltage * pf)

# 5. Cable
cable = Cable()
cable.setLengthM(cable_length)
cable.sizeCable(fla, voltage, cable_length, "Tray", 40.0)

# 6. Switchgear
sg = Switchgear()
sg.sizeSwitchgear(fla, motor.getRatedPowerKW(), voltage, True)

# 7. Transformer (for this single load)
apparent_power = vfd_input / pf
xfmr = Transformer()
xfmr.sizeTransformer(apparent_power, 11000.0, voltage)

# Print results
print("=" * 60)
print(f"  500 kW COMPRESSOR — COMPLETE DRIVE SYSTEM")
print("=" * 60)

print(f"\n  MOTOR ({motor.getEfficiencyClass()})")
print(f"    Rated:      {motor.getRatedPowerKW():.0f} kW, {voltage:.0f} V")
print(f"    Speed:      {motor.getRatedSpeedRPM():.0f} RPM ({motor.getPoles()}-pole)")
print(f"    Efficiency: {motor.getEfficiencyPercent():.1f}%")
print(f"    Frame:      {motor.getFrameSize()}, Weight: {motor.getWeightKg():.0f} kg")

print(f"\n  VFD ({vfd.getTopologyType()})")
print(f"    Topology:   {vfd.getTopologyType()}, {vfd.getPulseConfiguration()}")
print(f"    Efficiency: {vfd.getEfficiencyPercent():.1f}%")
print(f"    THD:        {vfd.getThdCurrentPercent():.1f}%")
print(f"    Input PF:   {vfd.getInputPowerFactor():.2f}")
print(f"    Cooling:    {vfd.getCoolingMethod()}")

print(f"\n  POWER CHAIN")
print(f"    Shaft power:    {shaft_power:>8.1f} kW")
print(f"    Motor input:    {motor_input:>8.1f} kW  (motor loss: {motor_loss:.1f} kW)")
print(f"    VFD input:      {vfd_input:>8.1f} kW  (VFD loss: {vfd_loss:.1f} kW)")
print(f"    Total loss:     {total_loss:>8.1f} kW")
print(f"    Overall eff:    {shaft_power / vfd_input * 100:.1f}%")
print(f"    Apparent power: {apparent_power:>8.1f} kVA")

print(f"\n  CABLE")
print(f"    Size:        {cable.getCrossSectionMM2():.0f} mm\u00b2, {cable_length:.0f} m")
print(f"    Ampacity:    {cable.getAmpacityA():.0f} A (load: {fla:.0f} A)")
print(f"    Voltage drop: {cable.getVoltageDropPercent():.2f}%")
print(f"    SC withstand: {cable.getShortCircuitWithstandKA():.1f} kA")

print(f"\n  SWITCHGEAR")
print(f"    Type:    {sg.getSwitchgearType()}, Starter: {sg.getStarterType()}")
print(f"    CB:      {sg.getCircuitBreakerRatingA():.0f} A")

print(f"\n  TRANSFORMER")
print(f"    Rating:  {xfmr.getRatedPowerKVA():.0f} kVA")
print(f"    Voltage: {xfmr.getPrimaryVoltageV():.0f}/{xfmr.getSecondaryVoltageV():.0f} V")
print(f"    Eff:     {xfmr.getEfficiencyPercent():.1f}%")
print(f"    Losses:  {xfmr.getTotalLossKW():.1f} kW")

In [ ]:
# Sankey-style power flow visualization (horizontal bar)
fig, ax = plt.subplots(figsize=(12, 4))

# Stacked horizontal bar showing power flow
categories = ["Shaft\nOutput", "Motor\nLoss", "VFD\nLoss"]
values = [shaft_power, motor_loss, vfd_loss]
colors_bar = ["#4e79a7", "#e15759", "#f28e2b"]

left = 0
for cat, val, col in zip(categories, values, colors_bar):
    ax.barh(0, val, left=left, height=0.5, color=col, edgecolor="white",
            label=f"{cat}: {val:.1f} kW")
    ax.text(left + val/2, 0, f"{val:.0f} kW", ha="center", va="center",
            fontsize=10, fontweight="bold", color="white")
    left += val

ax.set_xlim(0, vfd_input * 1.05)
ax.set_yticks([])
ax.set_xlabel("Power (kW)")
ax.set_title(f"Power Flow: {vfd_input:.0f} kW input \u2192 {shaft_power:.0f} kW shaft "
             f"(\u03b7 = {shaft_power/vfd_input*100:.1f}%)")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

1. **Motor sizing** — IEC standard power steps, efficiency classes IE1–IE4
2. **VFD topology** — automatic selection based on voltage/power
3. **Harmonics** — THD analysis per IEEE 519
4. **Efficiency maps** — combined motor + VFD at variable speed & load
5. **Cable sizing** — derating factors, voltage drop, short-circuit withstand
6. **Hazardous area** — IECEx/ATEX zone classification
7. **Complete drive system** — full chain analysis from transformer to shaft

### Key Equations

| Quantity | Formula |
|----------|--------|
| Full-load current | $I = P / (\sqrt{3} V \cos\varphi)$ |
| Motor efficiency | $\eta_m = P_{\text{shaft}} / P_{\text{elec}}$ |
| VFD efficiency | $\eta_{\text{vfd}} = P_{\text{motor}} / P_{\text{bus}}$ |
| Voltage drop | $\Delta V = \sqrt{3} I L (r\cos\varphi + x\sin\varphi)$ |
| SC withstand | $I_{sc} = kA / \sqrt{t}$ |

See also:
- `compressor_electrical_design.ipynb` — integrated process simulation workflow
- `process_plant_load_list.ipynb` — plant-wide load list and transformer sizing